# 🏢 Bureaux — Location Marrakech
## Nettoyage, Feature Engineering & Modélisation XGBoost

**Objectif** : prédire le loyer mensuel (MAD) d'un bureau à Marrakech.  
**Pipeline** : `pipeline/locations/pip_bureaux.py`  
**Données** : `data/marrakech_immo_location/bureaux_location.csv`


In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
ROOT = Path.cwd().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pipeline.locations.pip_bureaux import (
    load_data, split_and_encode, build_pipeline, train, evaluate,
    tune_hyperparams, plot_results, predict_price,
    DATA_PATH, MODEL_PATH,
    NUMERIC_FEATURES, BINARY_FEATURES, CATEGORICAL_FEATURES,
)

plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})
print("✅ Imports OK")
print(f"   DATA  : {DATA_PATH}")
print(f"   MODEL : {MODEL_PATH}")


## 1. Aperçu des données brutes

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Shape brut : {df_raw.shape}")
df_raw.head(3)


In [ ]:
print("Distribution type_bien :", df_raw["type_bien"].value_counts().to_string())
print("\nDistribution quartier :\n", df_raw["quartier"].value_counts().head(10).to_string())
print("\nNulls:", df_raw.isnull().sum()[df_raw.isnull().sum() > 0].to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
valid_prix = df_raw["prix_num"].dropna()
axes[0].hist(valid_prix[valid_prix < 50_000], bins=50, color="#2563eb", alpha=0.8, edgecolor="white")
axes[0].axvline(valid_prix.median(), color="red", ls="--", label=f"Médiane: {valid_prix.median():,.0f}")
axes[0].set_xlabel("Loyer (MAD/mois)"); axes[0].set_title("Distribution prix bruts (< 50K)"); axes[0].legend()

valid_surf = df_raw["surface_num"].dropna()
axes[1].hist(valid_surf[valid_surf < 500], bins=50, color="#10b981", alpha=0.8, edgecolor="white")
axes[1].axvline(valid_surf.median(), color="red", ls="--", label=f"Médiane: {valid_surf.median():.0f} m²")
axes[1].set_xlabel("Surface (m²)"); axes[1].set_title("Distribution surface brute (< 500 m²)"); axes[1].legend()
plt.tight_layout(); plt.show()


## 2. Nettoyage & Feature Engineering

In [ ]:
df = load_data(DATA_PATH)
print(f"\nShape après nettoyage : {df.shape}")
print(f"Loyer médian : {df['prix_num'].median():,.0f} MAD/mois")
print(f"Prix/m² médian : {df['prix_m2'].median():.1f} MAD/m²/mois")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Distributions après nettoyage — Bureaux", fontsize=14, fontweight="bold")

axes[0,0].hist(df["prix_num"], bins=50, color="#2563eb", alpha=0.8, edgecolor="white")
axes[0,0].set_title("Loyer mensuel (MAD)"); axes[0,0].set_xlabel("MAD/mois")

axes[0,1].hist(np.log(df["prix_num"]), bins=50, color="#7c3aed", alpha=0.8, edgecolor="white")
axes[0,1].set_title("log(Loyer)"); axes[0,1].set_xlabel("log(MAD)")

axes[0,2].hist(df["surface_num"], bins=50, color="#10b981", alpha=0.8, edgecolor="white")
axes[0,2].set_title("Surface (m²)"); axes[0,2].set_xlabel("m²")

axes[1,0].hist(df["prix_m2"], bins=50, color="#f59e0b", alpha=0.8, edgecolor="white")
axes[1,0].set_title("Prix/m²/mois (MAD)"); axes[1,0].set_xlabel("MAD/m²/mois")

df["zone_clean"].value_counts().head(10).plot(kind="barh", ax=axes[1,1], color="#2563eb")
axes[1,1].set_title("Top zones"); axes[1,1].invert_yaxis()

df["etage"].value_counts().sort_index().plot(kind="bar", ax=axes[1,2], color="#10b981")
axes[1,2].set_title("Distribution étages"); axes[1,2].tick_params(axis="x", rotation=0)

plt.tight_layout(); plt.show()


### 2.1 Loyer médian par zone

In [ ]:
zone_stats = df.groupby("zone_clean").agg(
    nb=("prix_num","count"),
    loyer_median=("prix_num","median"),
    pm2_median=("prix_m2","median"),
    surface_mediane=("surface_num","median"),
).sort_values("loyer_median", ascending=False)
print(zone_stats.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top = zone_stats[zone_stats["nb"] >= 5].sort_values("loyer_median")
axes[0].barh(top.index, top["loyer_median"] / 1e3, color="#2563eb", alpha=0.85)
axes[0].set_xlabel("Loyer médian (K MAD/mois)"); axes[0].set_title("Loyer médian par zone (≥5 annonces)")
axes[1].barh(top.index, top["pm2_median"], color="#10b981", alpha=0.85)
axes[1].set_xlabel("Prix/m²/mois (MAD)"); axes[1].set_title("Prix/m² médian par zone")
plt.tight_layout(); plt.show()


### 2.2 Keywords NLP & Équipements

In [ ]:
kw_cols = [c for c in BINARY_FEATURES if c.startswith("kw_")]
eq_cols = [c for c in BINARY_FEATURES if not c.startswith("kw_")]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df[kw_cols].sum().sort_values(ascending=True).plot(kind="barh", ax=axes[0], color="#7c3aed", alpha=0.85)
axes[0].set_title("Fréquence keywords NLP")
df[eq_cols].sum().sort_values(ascending=True).plot(kind="barh", ax=axes[1], color="#2563eb", alpha=0.85)
axes[1].set_title("Fréquence équipements")
plt.tight_layout(); plt.show()


## 3. Split train/test & Feature Engineering

In [ ]:
X_train, X_test, y_train, y_test, df_train, df_test, stats = split_and_encode(df, test_size=0.2, random_state=42)
print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"\nFeatures : {len(stats['feature_cols'])} total")
print(f"  Numériques    : {len(stats['numeric_cols'])}")
print(f"  Binaires      : {len(stats['binary_cols'])}")
print(f"  Catégorielles : {len(stats['categorical_cols'])}")


## 4. Entraînement XGBoost (défaut)

In [ ]:
pipeline_default = build_pipeline(stats)
pipeline_default = train(pipeline_default, X_train, y_train)
metrics_default = evaluate(pipeline_default, X_train, X_test, y_train, y_test)


## 5. Optimisation Hyperparamètres — Optuna

In [ ]:
best_params = tune_hyperparams(X_train, y_train, stats, n_trials=40)
print("\nMeilleurs hyperparamètres :")
for k, v in best_params.items():
    print(f"  {k}: {v}")


In [ ]:
pipeline_tuned = build_pipeline(stats, best_params)
pipeline_tuned = train(pipeline_tuned, X_train, y_train)
metrics_tuned  = evaluate(pipeline_tuned, X_train, X_test, y_train, y_test)
stats["mode"]  = metrics_tuned["mode"]


### 5.1 Comparaison défaut vs tuné

In [ ]:
import pandas as pd
comparison = pd.DataFrame({
    "Défaut": {k: metrics_default.get(k) for k in ["r2_test","mape","cv_mean"]},
    "Optuna": {k: metrics_tuned.get(k)   for k in ["r2_test","mape","cv_mean"]},
}).rename(index={"r2_test":"R² test","mape":"MAPE (%)","cv_mean":"CV R² (mean)"})
print(comparison.round(4).to_string())


## 6. Visualisations

In [ ]:
from pathlib import Path
save_dir = Path.cwd()
plot_results(pipeline_tuned, X_test, y_test, metrics_tuned, save_dir=save_dir)


### 6.1 Analyse des erreurs par zone

In [ ]:
import numpy as np
y_pred_te = pipeline_tuned.predict(X_test)
prix_reel = np.exp(y_test)
prix_pred = np.exp(y_pred_te)
pct_err   = np.abs((prix_pred - prix_reel) / prix_reel) * 100

df_err = df_test.copy()
df_err["pct_err"] = pct_err
err_by_zone = df_err.groupby("zone_clean").agg(
    n=("pct_err","count"),
    mape_zone=("pct_err","mean"),
    loyer_med=("prix_num","median"),
).sort_values("mape_zone", ascending=False)
print("Erreurs par zone (≥3 annonces) :")
print(err_by_zone[err_by_zone["n"] >= 3].round(1).to_string())


## 7. Prédictions exemples

In [ ]:
exemples = [
    {"surface_num": 50,  "zone_clean": "Guéliz",  "etage": 1, "parking": 0, "climatisation": 1,
     "titre": "Bureau climatisé Guéliz centre"},
    {"surface_num": 100, "zone_clean": "Guéliz",  "etage": 3, "parking": 1, "ascenseur": 1, "climatisation": 1,
     "titre": "Plateau bureau climatisé Guéliz angle"},
    {"surface_num": 200, "zone_clean": "gueliz",  "etage": 0, "parking": 1, "securite": 1,
     "titre": "Open space lumineux Guéliz"},
    {"surface_num": 70,  "zone_clean": "autre",   "etage": 0,
     "titre": "Bureau Marrakech périphérie"},
    {"surface_num": 150, "zone_clean": "hivernage","etage": 2, "parking": 1, "climatisation": 1, "securite": 1,
     "titre": "Bureau haut standing Hivernage résidence"},
]
for ex in exemples:
    predict_price(pipeline_tuned, ex, stats)


## 8. Sauvegarde du modèle

In [ ]:
import joblib
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"pipeline": pipeline_tuned, "stats": stats}, MODEL_PATH)
print(f"✅ Modèle sauvegardé : {MODEL_PATH}")

loaded = joblib.load(MODEL_PATH)
test_pred = np.exp(loaded["pipeline"].predict(X_test[:3]))
print(f"   Vérification — 3 prédictions : {test_pred.round(0)} MAD/mois")


## 9. Résumé

In [ ]:
print("=" * 55)
print("  RÉSUMÉ — BUREAUX LOCATION v1")
print("=" * 55)
print(f"  Données brutes         : {df_raw.shape[0]} annonces")
print(f"  Après nettoyage        : {df.shape[0]} annonces")
print(f"  Features totales       : {len(stats['feature_cols'])}")
print(f"  R² test (Optuna)       : {metrics_tuned['r2_test']:.3f}")
print(f"  MAPE test              : {metrics_tuned['mape']:.1f}%")
print(f"  CV R² (5-fold)         : {metrics_tuned['cv_mean']:.3f} ± {metrics_tuned['cv_std']:.3f}")
print(f"  MAD loyer              : {metrics_tuned['mad']:,.0f} MAD/mois")
print(f"  Mode prédiction        : {stats['mode']}")
print("=" * 55)
print()
print("Seuils de nettoyage :")
print("  - Prix    : 1 000 – 100 000 MAD/mois")
print("  - Surface : 10 – 2 000 m²")
print("  - Prix/m² : 10 – 2 000 MAD/m²/mois")
print()
print("Features clés :")
print("  - Target encoding par zone")
print("  - Surface relative dans la zone")
print("  - NLP : angle, centre, open_space, rénové")
print("  - Équipements : parking, ascenseur, clim, sécurité")
